In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import csv
import time
import chardet

import os

import PublicDataReader as pdr


D:\seoulmate\.venv\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


In [2]:
# 공간 데이터
space_file = r"../data/서울시_행정동_공간_base.csv"

# 이동 데이터
base_dir = r"D:\seoulmate\서울시_행정동_인구"
month_dirs = {
    "250_ORGN_CT_202408",
    "250_ORGN_CT_202409",
    "250_ORGN_CT_202410",
    "250_ORGN_CT_202411",
    "250_ORGN_CT_202412",
    "250_ORGN_CT_202501",
    "250_ORGN_CT_202502",
    "250_ORGN_CT_202503",
    "250_ORGN_CT_202504",
    "250_ORGN_CT_202505",
    "250_ORGN_CT_202506",
    "250_ORGN_CT_202507",
    "250_ORGN_CT_202508",
    "250_ORGN_CT_202509",
    "250_ORGN_CT_202510",
    "250_ORGN_CT_202511",
    "250_ORGN_CT_202512",
    "250_ORGN_CT_202601",
    "250_ORGN_CT_202602",
    "250_ORGN_CT_202603",
    "250_ORGN_CT_202604",
}


# 행정동ID-행정동코드-법정동코드 매핑

edm_mapping_name = r"../data/서울시_행정동ID_행정동코드_맵핑_base.csv"


monthly_population_name = r"../data/서울시_행정동_인구_월단위_202406_202604.csv"

daily_population_name = r"../data/서울시_행정동_인구_일단위_202406_202604.csv"



In [3]:

# 법정동 갯수는 467개
# 행정동 갯수는 426개

edm_mapping_df = pd.read_csv(edm_mapping_name, encoding="utf-8-sig")


print(f"맵핑 갯수:{len(edm_mapping_df)}, 행정동ID 갯수:{len(edm_mapping_df['행정동_ID'].unique())}, 행정동코드 갯수:{len(edm_mapping_df['행정동코드'].unique())}")

edm_mapping_df.head(2)

맵핑 갯수:426, 행정동ID 갯수:426, 행정동코드 갯수:426


,행정동_ID,행정동코드,행정동_명칭,자치구_명칭
0,11010720,1111051500,청운효자동,종로구
1,11010530,1111053000,사직동,종로구


In [15]:
# 1:1 인지 확인

#
# edm_mapping_df["cnt"] = edm_mapping_df.groupby("행정동_ID")["행정동코드"].transform("count")
#
# print(len(edm_mapping_df))
#
# dup_df = edm_mapping_df[edm_mapping_df["cnt"]!=1]
#
# print(len(dup_df))
#
# dup_df


In [5]:



columns = [
    "일자","시각","행정동코드","순위","대도시권거주지코드",
    "생활인구합계"
]


# 기존 spatiotemporal_name 삭제
if os.path.exists(daily_population_name):
    os.remove(daily_population_name)


# spatiotemporal_name 생성
for month_dir in sorted(month_dirs):

    people_dir = os.path.join(base_dir, month_dir)

    files = os.listdir(people_dir)

    for file in files:

        people_file = os.path.join(people_dir, file)

        print(f"{people_file} 처리")

        try:
            people_df = pd.read_csv(people_file,
                                    encoding="cp949",
                                    header=None,
                                    names=columns,
                                    usecols=range(len(columns)),
                                    dtype=str)
        except:
            people_df = pd.read_csv(people_file,
                                    encoding="utf-8-sig",
                                    header=None,
                                    names=columns,
                                    usecols=range(len(columns)),
                                    dtype=str)



        # 시작을 모르면 제외
        people_df = people_df[
            people_df["행정동코드"].notna() &
            (people_df["행정동코드"] != "-1")
        ]


        people_df = people_df[["일자","행정동코드","생활인구합계"]].copy()


        people_df["생활인구합계"] = pd.to_numeric(
           people_df["생활인구합계"],
            errors="coerce"
        )

        people_df["생활인구합계"] = (
            people_df["생활인구합계"]
            .fillna(0)
            .astype(int)
        )


        people_sum_df = people_df.groupby(["일자","행정동코드"],as_index=False).agg({
        "생활인구합계": "sum",
        })


        print(f"갯수:{len(people_sum_df)}")
        print(people_sum_df.head(2))

        people_sum_df.to_csv(
            daily_population_name,
            index=False,
            header=not os.path.exists(daily_population_name),
            mode="a",
            encoding="utf-8-sig")

#        break
#    break



D:\seoulmate\서울시_행정동_인구\250_ORGN_CT_202408\250_ORGN_CT_20240801.csv 처리
갯수:427
         일자     행정동코드  생활인구합계
0  20240801  11110515  345452
1  20240801  11110530  818959
D:\seoulmate\서울시_행정동_인구\250_ORGN_CT_202408\250_ORGN_CT_20240802.csv 처리
갯수:427
         일자     행정동코드  생활인구합계
0  20240802  11110515  350072
1  20240802  11110530  789719
D:\seoulmate\서울시_행정동_인구\250_ORGN_CT_202408\250_ORGN_CT_20240803.csv 처리
갯수:427
         일자     행정동코드  생활인구합계
0  20240803  11110515  352967
1  20240803  11110530  509701
D:\seoulmate\서울시_행정동_인구\250_ORGN_CT_202408\250_ORGN_CT_20240804.csv 처리
갯수:427
         일자     행정동코드  생활인구합계
0  20240804  11110515  322000
1  20240804  11110530  459308
D:\seoulmate\서울시_행정동_인구\250_ORGN_CT_202408\250_ORGN_CT_20240805.csv 처리
갯수:427
         일자     행정동코드  생활인구합계
0  20240805  11110515  327009
1  20240805  11110530  731552
D:\seoulmate\서울시_행정동_인구\250_ORGN_CT_202408\250_ORGN_CT_20240806.csv 처리
갯수:427
         일자     행정동코드  생활인구합계
0  20240806  11110515  316420
1  20240806  11110530 

In [6]:
monthly_people_df = pd.read_csv(daily_population_name, encoding="utf-8-sig")

In [7]:
# 월단위를 생성

monthly_people_df = pd.read_csv(daily_population_name, encoding="utf-8-sig")


monthly_people_df["YYYYMM"] = pd.to_datetime(
    monthly_people_df["일자"], format="%Y%m%d").dt.strftime("%Y%m")

index_cols = [
    "YYYYMM",
    "행정동코드"
]

sum_cols = [
    "생활인구합계"
]

monthly_people_df = monthly_people_df.groupby(index_cols,as_index=False)["생활인구합계"].sum()

monthly_people_df.to_csv(monthly_population_name, index=False, encoding="utf-8-sig")


print(f"갯수: {len(monthly_people_df)}")
monthly_people_df.head(2)


갯수: 8955


,YYYYMM,행정동코드,생활인구합계
0,202408,11110515,10466681
1,202408,11110530,21399917
